In [33]:
import pandas as pd
import numpy as np

In [35]:
from google.colab import files

uploaded = files.upload()
file_name = next(iter(uploaded))

df = pd.read_csv(file_name)
print(f"Dataset loaded successfully: {file_name}")

Saving Day8_Ecommerce_Sales_Dataset.csv to Day8_Ecommerce_Sales_Dataset.csv
Dataset loaded successfully: Day8_Ecommerce_Sales_Dataset.csv


In [36]:
print('Shape:', df.shape)
print('\nColumns:')
print(df.columns.tolist())

display(df.head())

Shape: (100, 13)

Columns:
['Order_ID', 'Order_Date', 'Customer', 'City', 'Region', 'Category', 'Product', 'Quantity', 'Unit_Price', 'Discount_Percent', 'Total_Sales', 'Rating', 'Payment_Method']


,Order_ID,Order_Date,Customer,City,Region,Category,Product,Quantity,Unit_Price,Discount_Percent,Total_Sales,Rating,Payment_Method
0,1001,2026-02-27,Kabir Ali,Hyderabad,South,Electronics,Wireless Headphones,3,1499,0,4497.00,3.8,Cash on Delivery
1,1002,2026-01-24,Aditya Verma,Jammu,North,Sports,Running Shoes,1,2799,0,2799.00,4.1,Cash on Delivery
2,1003,2026-04-18,Sara Ahmed,Chennai,South,Electronics,Power Bank,2,1199,15,2038.30,4.6,Cash on Delivery
3,1004,2026-03-13,Kabir Ali,Lucknow,North,Electronics,Mechanical Keyboard,4,2499,5,9496.20,4.1,Debit Card
4,1005,2026-03-30,Priya Menon,Jammu,North,Electronics,Smart Watch,1,3299,5,3134.05,4.2,UPI


In [37]:
df.columns = (
    df.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r'[^a-z0-9]+', '_', regex=True)
    .str.strip('_')
)

print('Standardized columns:')
print(df.columns.tolist())

def find_column(possible_names):
    for name in possible_names:
        if name in df.columns:
            return name
    for col in df.columns:
        if any(name in col for name in possible_names):
            return col
    return None

sales_col = find_column(['sales', 'total_sales', 'amount', 'revenue', 'total_amount', 'price'])
quantity_col = find_column(['quantity', 'qty', 'units'])
category_col = find_column(['category', 'product_category'])
city_col = find_column(['city', 'location'])
product_col = find_column(['product', 'product_name', 'item', 'item_name'])
payment_col = find_column(['payment_method', 'payment', 'payment_type', 'mode_of_payment'])
order_col = find_column(['order_id', 'orderid', 'order_no', 'order_number', 'id'])

field_map = {
    'Sales': sales_col,
    'Quantity': quantity_col,
    'Category': category_col,
    'City': city_col,
    'Product': product_col,
    'Payment Method': payment_col,
    'Order ID': order_col
}

display(pd.DataFrame({'Field': field_map.keys(), 'Detected Column': field_map.values()}))

Standardized columns:
['order_id', 'order_date', 'customer', 'city', 'region', 'category', 'product', 'quantity', 'unit_price', 'discount_percent', 'total_sales', 'rating', 'payment_method']


,Field,Detected Column
0,Sales,total_sales
1,Quantity,quantity
2,Category,category
3,City,city
4,Product,product
5,Payment Method,payment_method
6,Order ID,order_id


In [38]:
df.info()

missing = pd.DataFrame({
    'Missing Values': df.isna().sum(),
    'Missing %': (df.isna().mean() * 100).round(2)
})
display(missing[missing['Missing Values'] > 0])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          100 non-null    int64  
 1   order_date        100 non-null    object 
 2   customer          100 non-null    object 
 3   city              100 non-null    object 
 4   region            100 non-null    object 
 5   category          100 non-null    object 
 6   product           100 non-null    object 
 7   quantity          100 non-null    int64  
 8   unit_price        100 non-null    int64  
 9   discount_percent  100 non-null    int64  
 10  total_sales       100 non-null    float64
 11  rating            100 non-null    float64
 12  payment_method    100 non-null    object 
dtypes: float64(2), int64(4), object(7)
memory usage: 10.3+ KB


,Missing Values,Missing %


In [39]:
print('Single column example:')
display(df[[product_col]].head() if product_col else df.iloc[:, [0]].head())

selected_columns = [c for c in [order_col, product_col, category_col, city_col, sales_col, quantity_col, payment_col] if c]
print('Selected columns:')
display(df[selected_columns].head())

Single column example:


,product
0,Wireless Headphones
1,Running Shoes
2,Power Bank
3,Mechanical Keyboard
4,Smart Watch


Selected columns:


,order_id,product,category,city,total_sales,quantity,payment_method
0,1001,Wireless Headphones,Electronics,Hyderabad,4497.00,3,Cash on Delivery
1,1002,Running Shoes,Sports,Jammu,2799.00,1,Cash on Delivery
2,1003,Power Bank,Electronics,Chennai,2038.30,2,Cash on Delivery
3,1004,Mechanical Keyboard,Electronics,Lucknow,9496.20,4,Debit Card
4,1005,Smart Watch,Electronics,Jammu,3134.05,1,UPI


In [40]:
if sales_col:
    sales_numeric = pd.to_numeric(df[sales_col], errors='coerce')
    threshold = sales_numeric.quantile(0.75)
    high_value_orders = df[sales_numeric >= threshold].copy()
    print(f'High-value order threshold (75th percentile): {threshold:.2f}')
    display(high_value_orders.head(10))

if quantity_col:
    quantity_numeric = pd.to_numeric(df[quantity_col], errors='coerce')
    large_quantity_orders = df[quantity_numeric >= quantity_numeric.quantile(0.75)].copy()
    print('Orders in the top 25% by quantity:')
    display(large_quantity_orders.head(10))

High-value order threshold (75th percentile): 6427.56


,order_id,order_date,customer,city,region,category,product,quantity,unit_price,discount_percent,total_sales,rating,payment_method
3,1004,2026-03-13,Kabir Ali,Lucknow,North,Electronics,Mechanical Keyboard,4,2499,5,9496.20,4.1,Debit Card
9,1010,2026-03-11,Aman Kumar,Kochi,South,Clothing,Jacket,4,2499,10,8996.40,4.7,Credit Card
10,1011,2026-03-22,Rahul Das,Hyderabad,South,Home & Kitchen,Coffee Maker,2,3499,0,6998.00,4.2,UPI
12,1013,2026-03-09,Maryam Khan,Lucknow,North,Clothing,Jacket,5,2499,15,10620.75,4.8,Net Banking
13,1014,2026-02-05,Dev Patel,Jaipur,North,Sports,Running Shoes,3,2799,0,8397.00,4.6,UPI
16,1017,2026-03-17,Sana Malik,Hyderabad,South,Sports,Cricket Bat,3,2499,0,7497.00,4.0,Net Banking
18,1019,2026-05-19,Reyansh Jain,Chennai,South,Clothing,Jacket,3,2499,0,7497.00,4.7,UPI
32,1033,2026-06-28,Maryam Khan,Pune,West,Home & Kitchen,Mixer Grinder,3,2999,10,8097.30,4.7,Net Banking
37,1038,2026-03-08,Aditya Verma,Pune,West,Sports,Dumbbell Set,5,1999,5,9495.25,5.0,Debit Card
40,1041,2026-03-14,Ishita Gupta,Delhi,North,Home & Kitchen,Coffee Maker,2,3499,5,6648.10,4.6,Cash on Delivery


Orders in the top 25% by quantity:


,order_id,order_date,customer,city,region,category,product,quantity,unit_price,discount_percent,total_sales,rating,payment_method
3,1004,2026-03-13,Kabir Ali,Lucknow,North,Electronics,Mechanical Keyboard,4,2499,5,9496.20,4.1,Debit Card
7,1008,2026-04-27,Aman Kumar,Jammu,North,Clothing,Hoodie,4,1599,5,6076.20,4.3,Credit Card
9,1010,2026-03-11,Aman Kumar,Kochi,South,Clothing,Jacket,4,2499,10,8996.40,4.7,Credit Card
12,1013,2026-03-09,Maryam Khan,Lucknow,North,Clothing,Jacket,5,2499,15,10620.75,4.8,Net Banking
15,1016,2026-03-06,Manya Rao,Pune,West,Books,The Alchemist,4,499,15,1696.60,3.5,UPI
24,1025,2026-01-02,Rohan Mehta,Chennai,South,Sports,Football,5,799,0,3995.00,5.0,UPI
26,1027,2026-05-02,Sara Ahmed,Chandigarh,North,Sports,Football,5,799,15,3395.75,4.6,Net Banking
27,1028,2026-04-19,Sana Malik,Chandigarh,North,Electronics,Wireless Headphones,4,1499,5,5696.20,4.6,UPI
28,1029,2026-01-28,Sara Ahmed,Hyderabad,South,Electronics,Wireless Headphones,4,1499,5,5696.20,4.1,Credit Card
29,1030,2026-03-13,Karan Joshi,Bengaluru,South,Books,Python Programming,4,749,0,2996.00,4.1,UPI


In [41]:
if sales_col:
    df[sales_col] = pd.to_numeric(df[sales_col], errors='coerce')
    print('Top 10 orders by sales:')
    display(df.sort_values(sales_col, ascending=False).head(10))

    print('Bottom 10 orders by sales:')
    display(df.sort_values(sales_col, ascending=True).head(10))

Top 10 orders by sales:


,order_id,order_date,customer,city,region,category,product,quantity,unit_price,discount_percent,total_sales,rating,payment_method
48,1049,2026-03-02,Ishita Gupta,Mumbai,West,Sports,Running Shoes,5,2799,0,13995.00,4.0,Net Banking
52,1053,2026-01-08,Ananya Singh,Hyderabad,South,Sports,Running Shoes,5,2799,5,13295.25,4.2,Credit Card
84,1085,2026-01-28,Sana Malik,Hyderabad,South,Home & Kitchen,Coffee Maker,4,3499,10,12596.40,4.3,Net Banking
89,1090,2026-05-13,Aditya Verma,Lucknow,North,Sports,Cricket Bat,5,2499,0,12495.00,3.8,Credit Card
12,1013,2026-03-09,Maryam Khan,Lucknow,North,Clothing,Jacket,5,2499,15,10620.75,4.8,Net Banking
47,1048,2026-06-24,Sara Ahmed,Lucknow,North,Home & Kitchen,Coffee Maker,3,3499,0,10497.00,5.0,UPI
69,1070,2026-06-30,Aditya Verma,Delhi,North,Clothing,Jacket,4,2499,0,9996.00,3.8,Net Banking
79,1080,2026-05-25,Aditya Verma,Delhi,North,Clothing,Jacket,4,2499,0,9996.00,4.6,Debit Card
51,1052,2026-03-13,Rohan Mehta,Chandigarh,North,Electronics,Mechanical Keyboard,4,2499,5,9496.20,4.2,Debit Card
3,1004,2026-03-13,Kabir Ali,Lucknow,North,Electronics,Mechanical Keyboard,4,2499,5,9496.20,4.1,Debit Card


Bottom 10 orders by sales:


,order_id,order_date,customer,city,region,category,product,quantity,unit_price,discount_percent,total_sales,rating,payment_method
5,1006,2026-01-21,Manya Rao,Kochi,South,Books,The Alchemist,1,499,10,449.10,4.2,Cash on Delivery
62,1063,2026-04-17,Aman Kumar,Chandigarh,North,Books,The Alchemist,1,499,10,449.10,4.8,Credit Card
73,1074,2026-03-28,Alina Shah,Lucknow,North,Books,Atomic Habits,1,599,10,539.10,5.0,Net Banking
96,1097,2026-02-02,Arjun Nair,Mumbai,West,Clothing,T-Shirt,1,799,15,679.15,3.5,UPI
60,1061,2026-05-12,Alina Shah,Bengaluru,South,Books,Python Programming,1,749,5,711.55,5.0,Cash on Delivery
82,1083,2026-06-25,Priya Menon,Jammu,North,Sports,Football,1,799,5,759.05,3.8,Cash on Delivery
30,1031,2026-01-04,Rohan Mehta,Ahmedabad,West,Sports,Yoga Mat,1,899,15,764.15,4.1,Credit Card
42,1043,2026-05-22,Kabir Ali,Jaipur,North,Clothing,T-Shirt,1,799,0,799.00,4.2,Debit Card
20,1021,2026-01-18,Reyansh Jain,Chandigarh,North,Sports,Yoga Mat,1,899,10,809.10,4.7,Credit Card
77,1078,2026-01-05,Maryam Khan,Kolkata,East,Books,Machine Learning Basics,1,999,10,899.10,4.7,UPI


In [42]:
if sales_col:
    total_sales = df[sales_col].sum()
    average_sales = df[sales_col].mean()
    highest_sales = df[sales_col].max()
    lowest_sales = df[sales_col].min()
else:
    total_sales = average_sales = highest_sales = lowest_sales = np.nan

total_quantity = df[quantity_col].sum() if quantity_col else np.nan
number_of_orders = df[order_col].nunique() if order_col else len(df)

overall_stats = pd.DataFrame({
    'Statistic': ['Total Sales', 'Average Sales', 'Highest Sale', 'Lowest Sale', 'Total Quantity Sold', 'Number of Orders'],
    'Value': [total_sales, average_sales, highest_sales, lowest_sales, total_quantity, number_of_orders]
})
display(overall_stats)

,Statistic,Value
0,Total Sales,447262.9500
1,Average Sales,4472.6295
2,Highest Sale,13995.0000
3,Lowest Sale,449.1000
4,Total Quantity Sold,294.0000
5,Number of Orders,100.0000


In [43]:
if category_col and sales_col:
    category_analysis = df.groupby(category_col).agg(
        Total_Sales=(sales_col, 'sum'),
        Average_Sales=(sales_col, 'mean'),
        Highest_Sale=(sales_col, 'max'),
        Lowest_Sale=(sales_col, 'min'),
        Number_of_Orders=(sales_col, 'count')
    ).sort_values('Total_Sales', ascending=False)
    display(category_analysis)

,Total_Sales,Average_Sales,Highest_Sale,Lowest_Sale,Number_of_Orders
category,,,,,
Sports,136026.45,5441.058000,13995.00,759.05,25
Home & Kitchen,113329.90,5396.661905,12596.40,1188.30,21
Electronics,92086.00,4846.631579,9496.20,1708.10,19
Clothing,69028.20,4930.585714,10620.75,679.15,14
Books,36792.40,1752.019048,4495.00,449.10,21


In [44]:
if category_col and quantity_col:
    category_quantity = df.groupby(category_col).agg(
        Total_Quantity=(quantity_col, 'sum'),
        Average_Quantity=(quantity_col, 'mean')
    ).sort_values('Total_Quantity', ascending=False)
    display(category_quantity)

,Total_Quantity,Average_Quantity
category,,
Sports,79,3.160000
Books,60,2.857143
Home & Kitchen,59,2.809524
Electronics,57,3.000000
Clothing,39,2.785714


In [45]:
if city_col and sales_col:
    city_analysis = df.groupby(city_col).agg(
        Total_Sales=(sales_col, 'sum'),
        Average_Sales=(sales_col, 'mean'),
        Number_of_Orders=(sales_col, 'count')
    ).sort_values('Total_Sales', ascending=False)
    display(city_analysis)

,Total_Sales,Average_Sales,Number_of_Orders
city,,,
Hyderabad,59076.45,8439.492857,7
Chennai,47493.20,4749.320000,10
Lucknow,46194.65,7699.108333,6
Pune,41908.55,4190.855000,10
Mumbai,41457.80,5182.225000,8
Kochi,37257.50,4139.722222,9
Chandigarh,34539.65,4317.456250,8
Delhi,30882.80,5147.133333,6
Jaipur,25534.00,3647.714286,7


In [46]:
if product_col and sales_col:
    product_analysis = df.groupby(product_col).agg(
        Total_Sales=(sales_col, 'sum'),
        Average_Sales=(sales_col, 'mean'),
        Total_Quantity=(quantity_col, 'sum') if quantity_col else (sales_col, 'count'),
        Number_of_Orders=(sales_col, 'count')
    ).sort_values('Total_Sales', ascending=False)
    display(product_analysis.head(15))

,Total_Sales,Average_Sales,Total_Quantity,Number_of_Orders
product,,,,
Coffee Maker,51610.25,7372.892857,16,7
Jacket,47106.15,9421.230000,20,5
Cricket Bat,41483.40,8296.680000,18,5
Running Shoes,41285.25,8257.050000,15,5
Wireless Headphones,33877.40,4839.628571,23,7
Dumbbell Set,30184.90,6036.980000,16,5
Mechanical Keyboard,26489.40,8829.800000,11,3
Electric Kettle,26382.40,5276.480000,19,5
Mixer Grinder,25341.55,6335.387500,9,4


In [47]:
if payment_col and sales_col:
    payment_analysis = df.groupby(payment_col).agg(
        Total_Sales=(sales_col, 'sum'),
        Average_Sales=(sales_col, 'mean'),
        Number_of_Orders=(sales_col, 'count')
    ).sort_values('Total_Sales', ascending=False)
    display(payment_analysis)

,Total_Sales,Average_Sales,Number_of_Orders
payment_method,,,
Credit Card,106540.25,4842.738636,22
Debit Card,98064.90,4903.245000,20
Net Banking,96371.25,5668.897059,17
UPI,96163.25,3698.586538,26
Cash on Delivery,50123.30,3341.553333,15


In [48]:
if category_col and city_col and sales_col:
    category_city = df.groupby([category_col, city_col]).agg(
        Total_Sales=(sales_col, 'sum'),
        Number_of_Orders=(sales_col, 'count')
    ).sort_values('Total_Sales', ascending=False)
    display(category_city.head(20))

Total_Sales  Number_of_Orders
category       city                                     
Sports         Hyderabad      29288.85                 3
Clothing       Delhi          21590.00                 3
Electronics    Chandigarh     19689.40                 3
Home & Kitchen Hyderabad      19594.40                 2
               Pune           17707.90                 4
Sports         Jaipur         17398.80                 4
               Chennai        15548.20                 3
Clothing       Kochi          15453.00                 3
Home & Kitchen Chennai        14895.60                 3
Sports         Pune           14892.55                 2
               Mumbai         13995.00                 1
Home & Kitchen Mumbai         13044.15                 2
Sports         Lucknow        12495.00                 1
Clothing       Lucknow        10620.75                 1
Home & Kitchen Lucknow        10497.00                 1
Electronics    Hyderabad      10193.20                 2
               Mumbai          9895.00                 2
               Lucknow         9496.20                 1
Sports         Kochi           9495.25                 1
Clothing       Chennai         9015.10                 2

In [49]:
print('TOP PERFORMERS')
print('=' * 50)

if category_col and sales_col:
    top_category = df.groupby(category_col)[sales_col].sum().idxmax()
    print('Top category:', top_category)

if city_col and sales_col:
    top_city = df.groupby(city_col)[sales_col].sum().idxmax()
    print('Top city:', top_city)

if product_col and sales_col:
    top_product = df.groupby(product_col)[sales_col].sum().idxmax()
    print('Top product:', top_product)

if payment_col and sales_col:
    top_payment = df.groupby(payment_col)[sales_col].sum().idxmax()
    print('Top payment method:', top_payment)

TOP PERFORMERS
Top category: Sports
Top city: Hyderabad
Top product: Coffee Maker
Top payment method: Credit Card


In [50]:
if sales_col:
    print('Highest-value order:')
    display(df.loc[[df[sales_col].idxmax()]])

    print('Lowest-value order:')
    display(df.loc[[df[sales_col].idxmin()]])

Highest-value order:


,order_id,order_date,customer,city,region,category,product,quantity,unit_price,discount_percent,total_sales,rating,payment_method
48,1049,2026-03-02,Ishita Gupta,Mumbai,West,Sports,Running Shoes,5,2799,0,13995.0,4.0,Net Banking


Lowest-value order:


,order_id,order_date,customer,city,region,category,product,quantity,unit_price,discount_percent,total_sales,rating,payment_method
5,1006,2026-01-21,Manya Rao,Kochi,South,Books,The Alchemist,1,499,10,449.1,4.2,Cash on Delivery


In [51]:
print('MEANINGFUL OBSERVATIONS')
print('=' * 60)
print(f'1. The dataset contains {len(df):,} transaction rows and {number_of_orders:,} unique orders.')

if sales_col:
    print(f'2. Total sales are {total_sales:,.2f}, with an average order sales value of {average_sales:,.2f}.')
    print(f'3. The highest individual sale is {highest_sales:,.2f}, while the lowest is {lowest_sales:,.2f}.')

if quantity_col:
    print(f'4. A total of {total_quantity:,.0f} units were sold.')

if category_col and sales_col:
    category_totals = df.groupby(category_col)[sales_col].sum().sort_values(ascending=False)
    print(f'5. The top-performing category by total sales is {category_totals.index[0]}, generating {category_totals.iloc[0]:,.2f}.')

if product_col and sales_col:
    product_totals = df.groupby(product_col)[sales_col].sum().sort_values(ascending=False)
    print(f'6. The top-performing product by total sales is {product_totals.index[0]}, generating {product_totals.iloc[0]:,.2f}.')

if city_col and sales_col:
    city_totals = df.groupby(city_col)[sales_col].sum().sort_values(ascending=False)
    print(f'7. The city with the highest total sales is {city_totals.index[0]}, with {city_totals.iloc[0]:,.2f}.')

if payment_col and sales_col:
    payment_totals = df.groupby(payment_col)[sales_col].sum().sort_values(ascending=False)
    print(f'8. The most significant payment method by total sales is {payment_totals.index[0]}, contributing {payment_totals.iloc[0]:,.2f}.')

MEANINGFUL OBSERVATIONS
1. The dataset contains 100 transaction rows and 100 unique orders.
2. Total sales are 447,262.95, with an average order sales value of 4,472.63.
3. The highest individual sale is 13,995.00, while the lowest is 449.10.
4. A total of 294 units were sold.
5. The top-performing category by total sales is Sports, generating 136,026.45.
6. The top-performing product by total sales is Coffee Maker, generating 51,610.25.
7. The city with the highest total sales is Hyderabad, with 59,076.45.
8. The most significant payment method by total sales is Credit Card, contributing 106,540.25.
